# **R/S benchmark — PCE training**

This notebook **only** fits and validates the PCE. It reads the `dataset_unique_train` / `dataset_unique_val` files written by [`01_generate_dataset.ipynb`](01_generate_dataset.ipynb). Diagnostic plots (speed-up, KL-divergence check) are in [`02_plot_pce_validation.ipynb`](02_plot_pce_validation.ipynb).

## **1. Libraries**

In [1]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from functions import *
from UQpy.distributions import Normal, JointIndependent

/home/casa-wand/steam2tb/2024-1_victor_hugo_renata_maria/.venv/lib/python3.11/site-packages/UQpy/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## **2. Random variables and fixed parameters**

Must match [`01_generate_dataset.ipynb`](01_generate_dataset.ipynb). This only rebuilds the distribution object, it draws no new samples.

In [2]:
r_mean = 5.0
r_std  = 0.8
s_mean = 2.0
s_std  = 0.6

n_latent_samples = 2500    # must match stage 1 — it is the filename prefix
n_lambdas        = 4
max_degree       = 5        # maximum total degree of the PCE polynomial basis

r_dist = Normal(loc=r_mean, scale=r_std)
s_dist = Normal(loc=s_mean, scale=s_std)
joint  = JointIndependent(marginals=[r_dist, s_dist])

## **3. Time grid**

Must match times written by [`01_generate_dataset.ipynb`](01_generate_dataset.ipynb).

In [3]:
times = np.linspace(0, 100, 5, endpoint=True)
times

array([  0.,  25.,  50.,  75., 100.])

## **4. Load the datasets and train the PCE at each time step**

In [4]:
print("="*60)
print("TRAINING THE BENCHMARK PCE")
print("="*60)

results = []
for t in times:
    with open(f'{n_latent_samples}_dataset_unique_train_{t}_benchmark.pkl', 'rb') as f:
        df_unique_train = dill.load(f)
    with open(f'{n_latent_samples}_dataset_unique_val_{t}_benchmark.pkl', 'rb') as f:
        df_unique_val = dill.load(f)

    result = train_and_validate_pce_from_dataset_benchmark(
                                                              df_unique_train=df_unique_train,
                                                              df_unique_val=df_unique_val,
                                                              joint=joint,
                                                              time_step=t,
                                                              n_latent_samples=n_latent_samples,
                                                              n_lambdas=n_lambdas,
                                                              max_degree=max_degree,
                                                              output_dir='.',
                                                          )
    result['x_train'] = df_unique_train[['r', 's']].to_numpy()
    results.append(result)

TRAINING THE BENCHMARK PCE

----------------------------------------
TRAINING PCE FOR TIME STEP: 0.0 years
----------------------------------------
1. PCE training dataset has been saved!
2. PCE statistcs has been saved!

----------------------------------------
TRAINING PCE FOR TIME STEP: 25.0 years
----------------------------------------
1. PCE training dataset has been saved!
2. PCE statistcs has been saved!

----------------------------------------
TRAINING PCE FOR TIME STEP: 50.0 years
----------------------------------------
1. PCE training dataset has been saved!
2. PCE statistcs has been saved!

----------------------------------------
TRAINING PCE FOR TIME STEP: 75.0 years
----------------------------------------
1. PCE training dataset has been saved!
2. PCE statistcs has been saved!

----------------------------------------
TRAINING PCE FOR TIME STEP: 100.0 years
----------------------------------------
1. PCE training dataset has been saved!
2. PCE statistcs has been saved

## **5. Validation summary**

How well the PCE reproduces each lambda, per time step.

In [5]:
validation_summary = pd.concat([r['statistics'] for r in results], ignore_index=True)
validation_summary.insert(0, 'Time (years)', [r['time_step'] for r in results])
validation_summary

,Time (years),MSE λ1,MSE λ2,MSE λ3,MSE λ4,R² λ1,R² λ2,R² λ3,R² λ4
0,0.0,0.000032,0.036096,0.000452,0.000413,0.999969,0.979504,0.002202,0.100693
1,25.0,0.000032,0.051272,0.000454,0.000469,0.999965,0.976717,0.065788,0.002610
2,50.0,0.000026,0.049197,0.000438,0.000383,0.999967,0.984233,0.015658,0.079338
3,75.0,0.000025,0.056486,0.000400,0.000399,0.999962,0.986619,-0.001781,0.060296
4,100.0,0.000022,0.063286,0.000419,0.000420,0.999960,0.989090,-0.018763,-0.008939
